# Customer Personality Segmentation
### MIT Applied Data Science & Machine Learning Certification

---

**Techniques used:** K-Means Clustering, Feature Scaling, Elbow Method, Silhouette Analysis  
**Libraries:** pandas, numpy, scikit-learn, matplotlib, seaborn, yellowbrick  
**Dataset:** 2,240 customers × 28 features (demographics, spending, campaign response)

---

## 1. Business Context & Objective

Understanding customer personality and behavior is pivotal for businesses to enhance customer satisfaction and increase revenue. Segmentation based on a customer's personality, demographics, and purchasing behavior allows companies to create tailored marketing campaigns, improve customer retention, and optimize product offerings.

A leading retail company with a rapidly growing customer base seeks to gain deeper insights into their customers' profiles. The company recognizes that understanding customer personalities, lifestyles, and purchasing habits can unlock significant opportunities for personalizing marketing strategies and creating loyalty programs.

**Goal:** Identify distinct customer segments so the business can:
- Develop **personalized marketing campaigns** to increase conversion rates
- Create effective **retention strategies** for high-value customers
- Optimize **resource allocation** — inventory, pricing, and store layouts

---

## 2. Data Dictionary

| Category | Feature | Description |
|---|---|---|
| **Customer Info** | `ID` | Unique customer identifier |
| | `Year_Birth` | Customer's year of birth |
| | `Education` | Education level |
| | `Marital_Status` | Marital status |
| | `Income` | Yearly household income (USD) |
| | `Kidhome` | Number of children in household |
| | `Teenhome` | Number of teenagers in household |
| | `Dt_Customer` | Date of enrollment with the company |
| | `Recency` | Days since last purchase |
| | `Complain` | Complained in last 2 years (1=Yes, 0=No) |
| **Spending (2yr)** | `MntWines` | Amount spent on wine |
| | `MntFruits` | Amount spent on fruits |
| | `MntMeatProducts` | Amount spent on meat |
| | `MntFishProducts` | Amount spent on fish |
| | `MntSweetProducts` | Amount spent on sweets |
| | `MntGoldProds` | Amount spent on gold products |
| **Campaigns** | `AcceptedCmp1–5` | Response to campaigns 1–5 (1=Yes, 0=No) |
| | `Response` | Response to last campaign |
| | `NumDealsPurchases` | Purchases made using a discount |
| **Shopping** | `NumWebPurchases` | Purchases via website |
| | `NumCatalogPurchases` | Purchases via catalog |
| | `NumStorePurchases` | Purchases in-store |
| | `NumWebVisitsMonth` | Website visits in last month |

---

## 3. Setup & Imports

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist, pdist

# Cluster visualization
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

import warnings
warnings.filterwarnings("ignore")

## 4. Load Data

In [ ]:
# Mount Google Drive (Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load data — tab-separated file
data = pd.read_csv("/content/Customer_Personality_Segmentation.csv", sep="\t")
df = data.copy()

print(f"Dataset shape: {df.shape}")
df.head()

---
## 5. Exploratory Data Analysis (EDA)

### 5.1 Data Types & Structure

In [ ]:
df.info()

> **Observations:**
> - The dataset contains **2,240 rows and 28 columns**.
> - Most columns are `int64`, with `Income` as `float64` and three object (categorical) columns: `Education`, `Marital_Status`, and `Dt_Customer`.
> - `Income` is the only column with missing values.

### 5.2 Statistical Summary

In [ ]:
df.describe().T

> **Observations:**
> - The **average household income is ~$52,247**.
> - `Year_Birth` ranges from 1893 to 1996 — potential outliers at the lower end worth investigating.
> - Spending columns (e.g. `MntWines`, `MntMeatProducts`) are highly right-skewed, indicating a small group of high-spending customers.

### 5.3 Missing Values

In [ ]:
df.isnull().sum()

> **Observations:** There are **24 missing values** in `Income`. Before choosing an imputation method, we check for skewness — if skewed, median is preferred over mean to avoid distortion from outliers.

In [ ]:
skew = df['Income'].skew()
print(f"Income skewness: {skew:.2f}")

> **Decision:** Skewness is above 1.0, confirming a right-skewed distribution. We use **median imputation** as it is more robust to outliers than the mean.

In [ ]:
# Impute missing income with the median
df['Income'] = df['Income'].fillna(df['Income'].median())

# Confirm no missing values remain
print(f"Missing values remaining: {df.isnull().sum().sum()}")

### 5.4 Duplicate Check

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

> **Observations:** No duplicate rows found. The dataset is clean and ready for further analysis.

### 5.5 Univariate Analysis

In [ ]:
numeric_data = df.select_dtypes(include='number')

numeric_data.hist(figsize=(20, 20), bins=20, edgecolor='black')
plt.suptitle('Distribution of All Numeric Features', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 5.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(40, 40))
sns.heatmap(numeric_data.corr(), annot=True, fmt='0.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix', fontsize=20)
plt.show()

> **Observations:**
> - `Income` shows strong positive correlation with spending across all product categories.
> - `NumCatalogPurchases` and `MntMeatProducts` are notably correlated — high-income customers tend to buy premium products via catalog.
> - Campaign acceptance features (`AcceptedCmp1–5`) are moderately correlated with each other, suggesting a consistent group of campaign-responsive customers.

---
## 6. K-Means Clustering

### 6.1 Feature Scaling

K-Means is distance-based, so all features must be on the same scale. We apply **StandardScaler** (z-score normalization).

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

scaler = StandardScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)

# Keep a copy to store cluster labels
data_scaled_copy = data_scaled.copy(deep=True)

print(f"Scaled data shape: {data_scaled.shape}")
data_scaled.head()

### 6.2 Choosing K — Elbow Method (WCSS)

In [ ]:
WCSS = {}

for k in range(1, 10):
    kmeans = KMeans(n_clusters=k, max_iter=1000, random_state=42).fit(data_scaled)
    WCSS[k] = kmeans.inertia_

plt.figure(figsize=(8, 5))
plt.plot(list(WCSS.keys()), list(WCSS.values()), 'bx-')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")
plt.title("Elbow Method — Optimal K Selection")
plt.xticks(range(1, 10))
plt.show()

> **Observations:** The elbow point occurs at **K=3**, where the rate of WCSS reduction begins to flatten. This suggests 3 clusters as the optimal value.

### 6.3 Validating K — Silhouette Score

In [ ]:
sc = {}

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42).fit(data_scaled)
    labels = kmeans.predict(data_scaled)
    sc[k] = silhouette_score(data_scaled, labels)

plt.figure(figsize=(8, 5))
plt.plot(list(sc.keys()), list(sc.values()), 'bx-')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score — Higher is Better")
plt.xticks(range(2, 8))
plt.show()

best_k = max(sc, key=sc.get)
print(f"Best K by Silhouette Score: {best_k} (score: {sc[best_k]:.4f})")

> **Observations:** The Silhouette Score confirms **K=3** as the optimal number of clusters. Both the Elbow Method and Silhouette Score are in agreement, giving us high confidence in this choice.

### 6.4 Final Model — K=3

In [ ]:
# Fit final model
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(data_scaled)

# Assign cluster labels
data_scaled_copy['Labels'] = kmeans.predict(data_scaled)
data['Labels'] = kmeans.predict(data_scaled)

# Cluster distribution
print("Cluster Sizes:")
print(data['Labels'].value_counts().sort_index())

---
## 7. Cluster Profiling

### 7.1 Mean & Median by Cluster

In [ ]:
mean = data.groupby('Labels')[numeric_cols].mean()
median = data.groupby('Labels')[numeric_cols].median()

# Combine and format
df_kmeans = pd.concat([mean, median], axis=0)
df_kmeans.index = [
    [f'Cluster {i} Mean' for i in mean.index] +
    [f'Cluster {i} Median' for i in median.index]
]

df_kmeans.T

### 7.2 Boxplot Profiling

In [ ]:
data_scaled_copy.boxplot(by='Labels', layout=(6, 6), figsize=(30, 20))
plt.suptitle('Feature Distribution by Cluster', fontsize=16)
plt.tight_layout()
plt.show()

> **Cluster Profiles:**
> - **Cluster 0 — Budget Shoppers:** Lower income, minimal spending across all categories, higher number of web visits but low online purchase conversion.
> - **Cluster 1 — Mid-Tier Customers:** Average income and spending, moderate campaign response, balanced mix of in-store and online purchases.
> - **Cluster 2 — Premium Customers:** High income, high spending on wines and meat, strong catalog purchase behavior, most responsive to marketing campaigns.

---
## 8. Business Recommendations

Based on the segmentation analysis, here are targeted recommendations for each cluster:

### 🟢 Cluster 0 — Budget Shoppers
- **Channel:** Focus on web and in-store discounts — they visit the website frequently but don't convert.
- **Action:** Introduce a **deals & loyalty points program** to incentivize online purchases.
- **Campaign:** Price-sensitive promotions (e.g., flash sales, bundle offers).

### 🟡 Cluster 1 — Mid-Tier Customers
- **Channel:** Mixed — both web and store. Retargeting campaigns can be effective here.
- **Action:** Upsell through **personalized product recommendations** based on past purchases.
- **Campaign:** Moderate-value campaigns with clear value propositions.

### 🔵 Cluster 2 — Premium Customers
- **Channel:** Catalog and in-store. These customers respond well to exclusive, curated offers.
- **Action:** Invest in a **VIP/loyalty program** — this group drives disproportionate revenue.
- **Campaign:** Premium product launches, early access, and personalized outreach.

### 🌐 Cross-Cluster Insight — Improve Online Conversion
The data shows strong website visit numbers across all segments but relatively low online purchase rates compared to in-store. The customer base skews older (born 1960s–1980s), suggesting **trust and UX** are likely barriers. Recommended actions:
- Simplify the checkout process
- Add trust signals (reviews, secure payment badges)
- Offer first-time online purchase incentives

---

## 9. Summary

| Step | Method | Key Finding |
|---|---|---|
| Missing values | Median imputation | 24 missing `Income` values treated |
| Scaling | StandardScaler | All features normalized before clustering |
| Optimal K | Elbow + Silhouette | Both methods confirmed **K=3** |
| Segmentation | K-Means | 3 distinct customer personas identified |
| Recommendations | Cluster profiling | Targeted strategies for each segment |

This project demonstrates the end-to-end application of unsupervised machine learning to solve a real business problem — transforming raw customer data into actionable segmentation strategy.